# Explanations: what the app will say about each lounge and area

Each decision (`features/decisions.ipynb`) gets a pyramid: a one-sentence answer, then the arguments that
matter most for it (ranked by `src.explain.prioritize`, never by the writer), each backed by 2-5
numbers copied from the decision's fact sheet. `src.explain.verify` rejects any explanation that
cites a value or writes a number not on the fact sheet.

The **AI** explanations here were written in a Claude Code session from the same prompts
`just explain` sends to the API (`python -m src.explain prompts|check|ingest`), so no API credits
were spent; they are cached in `data/explanations/cache.json` by a hash of their fact sheet, so a
changed number falls back to the deterministic **template**. The NOT SCORED airport lounge and
SKIP areas always use the template.

Data: `src.explain.v3_subjects()` (baseline, medium levels).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from collections import Counter

import pandas as pd
from IPython.display import Markdown, display

from src.explain import (LOUNGE_TABLE, AREA_TABLE, explain, load_cache, table_rows,
                         v3_subjects, verify)

pd.set_option("display.max_colwidth", 120)
cache = load_cache()
subjects = v3_subjects()
exps = {(k, sid): explain(k, sid, a, f, cache=cache) for k, sid, a, f in subjects}
facts = {(k, sid): f for k, sid, a, f in subjects}
print(Counter((k, e.source) for (k, _), e in exps.items()))
print("grounding failures:", sum(bool(verify(e, facts[key], key[0])) for key, e in exps.items()))

Counter({('area', 'template'): 561, ('lounge', 'template'): 24, ('uae', 'template'): 1})
grounding failures: 0


In [2]:
def show(kind, sid):
    e, f = exps[(kind, sid)], facts[(kind, sid)]
    lines = [f"### {sid} — **{e.action}** ({e.source})", "", f"**{e.headline}**", ""]
    for r in e.reasons:
        ev = "; ".join(f"{x.label}: {x.value}" for x in r.evidence)
        lines += [f"- **{r.topic}**: {r.claim}  \n  <small>{ev}</small>"]
    lines += ["", f"_{e.table_caption}_"]
    display(Markdown("\n".join(lines)))


def table(kind, sid):
    fields = {"lounge": LOUNGE_TABLE, "area": AREA_TABLE}[kind]
    return pd.DataFrame(table_rows(fields, facts[(kind, sid)])).set_index("Factor")

## The network summary (the app opens with this)

In [3]:
show("uae", "uae")

### uae — **SUMMARY** (template)

**3 PROTECT, 13 HOLD and 7 SHRINK across 23 scored lounges, with 49 areas to GROW.**

- **shrink**: Investigate before shrinking: al-barsha, city-walk, delma, jumeirah-park, khaleej-al-arabi, noya-plaza, westyas.  
  <small>SHRINK: 7; SHRINK lounges: al-barsha, city-walk, delma, jumeirah-park, khaleej-al-arabi, noya-plaza, westyas; Strongest SHRINK composite: 0.3271</small>
- **grow**: 49 areas to GROW, with 960,330 women 15+ between them.  
  <small>GROW: 49; GROW areas: al-aamerah-abu-dhabi, al-awir-dubai, al-batayih-sharjah, al-dhahir-abu-dhabi, al-dhaid-sharjah, al-dhait-south-ras-al-khaimah, al-fahlain-ras-al-khaimah, al-hamriya-sharjah, al-helio-ajman, al-hiyar-abu-dhabi, al-jerf-ajman, al-khibeesi-abu-dhabi, al-madam-sharjah, al-maqam-abu-dhabi, al-muwaihat-ajman, al-muwaij-i-abu-dhabi, al-nahdah-abu-dhabi, al-rahbah-abu-dhabi, al-ramlah-sharjah, al-rashidiya-ajman, al-rawdah-al-sharqiyah-abu-dhabi, al-sad-abu-dhabi, al-sader-abu-dhabi, al-samhah-abu-dhabi, al-shahamah-abu-dhabi, al-wathbah-abu-dhabi, bani-yas-abu-dhabi, dibba-al-hisn-sharjah, digdaga-ras-al-khaimah, falaj-hazza-abu-dhabi, ghalilah-ras-al-khaimah, hamidiya-ajman, kalba-sharjah, khatt-ras-al-khaimah, khor-fakkan-sharjah, manama-ajman, mazyad-abu-dhabi, mbazzarah-al-khadra-abu-dhabi, meshairef-ajman, milehah-sharjah, mudayrah-ajman, muntazy-ajman, mzeer-ah-abu-dhabi, rimah-abu-dhabi, sharjah-sharjah, shiab-al-ashkhar-abu-dhabi, um-ghafah-abu-dhabi, wasit-sharjah, zakhir-abu-dhabi; Women in GROW areas: 960330; Closest GROW area to a lounge: 0.9589; Furthest GROW area from a lounge: 54.9537</small>
- **protect**: Protect al-ain, al-dhafra, khalifa-city-a.  
  <small>PROTECT: 3; PROTECT lounges: al-ain, al-dhafra, khalifa-city-a; Weakest PROTECT composite: 0.667</small>
- **confidence**: 13 lounges sit in the HOLD band and 8 calls are low confidence.  
  <small>HOLD: 13; Low-confidence calls: 8; Lounges scored: 23</small>
- **data_gaps**: No revenue or lease data: these are location and market calls, not return-on-capital calls.  
  <small>Revenue data: False; Lease data: False; Salons searched: 3833</small>

_Lounge calls come from a fixed-scale scorecard; growth-area calls from two tests, size and premium saturation._

## Lounges, PROTECT first

In [4]:
order = {"PROTECT": 0, "HOLD": 1, "SHRINK": 2, "NOT SCORED": 3}
for k, sid, a, f in sorted((s for s in subjects if s[0] == "lounge"), key=lambda s: (order[s[2]], s[1])):
    show(k, sid)

### al-ain — **PROTECT** (template)

**PROTECT: composite 0.72 (PROTECT at 0.65 or more, SHRINK at 0.35 or less).**

- **demand**: Women served (Huff) is a strength: 18,652 women.  
  <small>Women served (Huff): 18651.8397; Addressable women 15+: 102438.9547; Women 15+ in catchment: 102438.9547; Affluence coverage: 0.0; Demand score: 0.9326</small>
- **cost**: Cost efficiency vs emirate is middling: n/a (missing).  
  <small>Cost efficiency vs emirate: None; Women per AED 1k of site cost: None; Site cost: None; Shop rent at the site: None; Cost score: 0.5</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

### al-dhafra — **PROTECT** (template)

**PROTECT: composite 0.94 (PROTECT at 0.65 or more, SHRINK at 0.35 or less).**

- **cost**: Cost efficiency vs emirate is a strength: 4.28 x median lounge.  
  <small>Cost efficiency vs emirate: 4.2833; Women per AED 1k of site cost: 65.3093; Site cost: 270.0; Shop rent at the site: 1200.0; Cost score: 1.0</small>
- **demand**: Women served (Huff) is a strength: 17,634 women.  
  <small>Women served (Huff): 17633.5017; Addressable women 15+: 22927.3037; Women 15+ in catchment: 22927.3037; Affluence coverage: 0.0; Demand score: 0.8817</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

### khalifa-city-a — **PROTECT** (template)

**PROTECT: composite 0.67 (PROTECT at 0.65 or more, SHRINK at 0.35 or less).**

- **demand**: Women served (Huff) is a strength: 16,679 women.  
  <small>Women served (Huff): 16679.2576; Addressable women 15+: 91431.5935; Women 15+ in catchment: 91431.5935; Affluence coverage: 0.0; Demand score: 0.834</small>
- **cost**: Cost efficiency vs emirate is middling: n/a (missing).  
  <small>Cost efficiency vs emirate: None; Women per AED 1k of site cost: None; Site cost: None; Shop rent at the site: None; Cost score: 0.5</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

### al-falah — **HOLD** (template)

**HOLD: composite 0.49 (PROTECT at 0.65 or more, SHRINK at 0.35 or less).**

- **demand**: Women served (Huff) is middling: 9,510 women.  
  <small>Women served (Huff): 9509.6479; Addressable women 15+: 49946.8292; Women 15+ in catchment: 49946.8292; Affluence coverage: 0.0; Demand score: 0.4755</small>
- **cost**: Cost efficiency vs emirate is middling: n/a (missing).  
  <small>Cost efficiency vs emirate: None; Women per AED 1k of site cost: None; Site cost: None; Shop rent at the site: None; Cost score: 0.5</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

### al-jada — **HOLD** (template)

**HOLD: composite 0.40 (PROTECT at 0.65 or more, SHRINK at 0.35 or less).**

- **demand**: Women served (Huff) is a weakness: 5,853 women.  
  <small>Women served (Huff): 5852.9265; Addressable women 15+: 141329.5351; Women 15+ in catchment: 141329.5351; Affluence coverage: 0.0; Demand score: 0.2926</small>
- **cost**: Cost efficiency vs emirate is middling: n/a (missing).  
  <small>Cost efficiency vs emirate: None; Women per AED 1k of site cost: None; Site cost: None; Shop rent at the site: None; Cost score: 0.5</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

...

### jumeirah-park — **SHRINK** (template)

**SHRINK: composite 0.20 (PROTECT at 0.65 or more, SHRINK at 0.35 or less).**

- **demand**: Women served (Huff) is a weakness: 1,636 women.  
  <small>Women served (Huff): 1636.4881; Addressable women 15+: 171594.8075; Women 15+ in catchment: 171594.8075; Affluence coverage: 0.9574; Demand score: 0.0818</small>
- **cost**: Cost efficiency vs emirate is a weakness: 0.82 x median lounge.  
  <small>Cost efficiency vs emirate: 0.8233; Women per AED 1k of site cost: 4.014; Site cost: 407.7; Shop rent at the site: 2118.0; Cost score: 0.3233</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

### khaleej-al-arabi — **SHRINK** (template)

**SHRINK: composite 0.33 (PROTECT at 0.65 or more, SHRINK at 0.35 or less).**

- **demand**: Women served (Huff) is a weakness: 3,083 women.  
  <small>Women served (Huff): 3083.1893; Addressable women 15+: 69366.5903; Women 15+ in catchment: 69366.5903; Affluence coverage: 0.0; Demand score: 0.1542</small>
- **cost**: Cost efficiency vs emirate is middling: n/a (missing).  
  <small>Cost efficiency vs emirate: None; Women per AED 1k of site cost: None; Site cost: None; Shop rent at the site: None; Cost score: 0.5</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

### noya-plaza — **SHRINK** (template)

**SHRINK: composite 0.03 (PROTECT at 0.65 or more, SHRINK at 0.35 or less).**

- **cost**: Cost efficiency vs emirate is a weakness: 0.34 x median lounge.  
  <small>Cost efficiency vs emirate: 0.3366; Women per AED 1k of site cost: 5.1323; Site cost: 270.0; Shop rent at the site: 1200.0; Cost score: 0.0</small>
- **demand**: Women served (Huff) is a weakness: 1,386 women.  
  <small>Women served (Huff): 1385.7164; Addressable women 15+: 45036.5814; Women 15+ in catchment: 45036.5814; Affluence coverage: 0.0; Demand score: 0.0693</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

### westyas — **SHRINK** (template)

**SHRINK: composite 0.21 (PROTECT at 0.65 or more, SHRINK at 0.35 or less).**

- **demand**: Women served (Huff) is a weakness: 3,160 women.  
  <small>Women served (Huff): 3160.0923; Addressable women 15+: 42368.0713; Women 15+ in catchment: 42368.0713; Affluence coverage: 0.0; Demand score: 0.158</small>
- **cost**: Cost efficiency vs emirate is a weakness: 0.77 x median lounge.  
  <small>Cost efficiency vs emirate: 0.7676; Women per AED 1k of site cost: 11.704; Site cost: 270.0; Shop rent at the site: 1200.0; Cost score: 0.2676</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

### zayed-international-airport — **NOT SCORED** (template)

**Not scored: this lounge serves travellers, not the women living around it, so the catchment model doesn't apply.**

- **demand**: Women served (Huff): n/a (missing), shown but not scored.  
  <small>Women served (Huff): None; Addressable women 15+: 75830.1549; Women 15+ in catchment: 75830.1549; Affluence coverage: 0.0</small>
- **capture**: Capture: 4% of catchment, shown but not scored.  
  <small>Capture: 0.0392; Premium substitutes: 11; Premium salons in catchment: 51; Thin premium market: False</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

The factor table the app shows next to a lounge's pyramid, for one example:

In [5]:
table("lounge", "al-barsha")

,Value,Unit,Threshold,What it means
Factor,,,,
Women served (Huff),"2,180",women,"Demand score: 0 women → 0, 20,000 women → 1; weight 1",Addressable women this lounge serves: its Huff share of every cell it reaches within 20 min.
Absorbed by siblings,1%,% of served,—,Share of its served women other lounges take over if it closes.
Women 15+ in catchment,"272,371",women,—,"Women aged 15+ living in the ~2 km cells within a 15-min drive (typical midday traffic). WorldPop adults, rebalanced..."
Addressable women 15+,"272,371",women,"GROW needs 20,000","Catchment women weighted by how affluent their cell is (observed median rent vs the women-weighted median, to the af..."
Affluence rent,"115,000",AED/yr,—,"Median annual household rent (DLD Ejari contracts, Jul-Oct 2026), weighted by women, over the cells with an observed..."
Affluence coverage,99%,% of women,"Under 50%: affluence mostly unknown, weighted neutral",Share of the women here living in cells with an observed rent. The rest are weighted neutral: affluence unknown. Dub...
Cells in catchment,64,count,—,Populated ~2 km grid cells whose centre is within the drive time.
Shared catchment,86%,% of catchment women,—,Share of the catchment's women who live in cells another open Bedashing lounge also reaches: the cannibalisation mea...
Premium salons in catchment,228,count,—,"Google Places salons in the catchment that count as premium: Google price 'expensive' or above, or (unpriced) at lea..."


## Growth areas: GROW, then WATCH by women

In [6]:
areas = sorted((s for s in subjects if s[0] == "area" and s[2] != "SKIP"),
               key=lambda s: (s[2] != "GROW", -s[3]["women"]))
for k, sid, a, f in areas:
    show(k, sid)

### sharjah-sharjah — **GROW** (template)

**GROW: big enough, and unsaturated.**

- **size**: About 281,946 addressable women 15+ live here, 281,946 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 281946.4916; Women 15+: 281946.4916; Size line: 20000; Big enough: True; Affluence coverage: 0.0</small>
- **saturation**: 116 premium reviews per 1k women, under the 150 line.  
  <small>Premium saturation: 116.4158; Premium salons: 107; Unsaturated: True; Saturation line: 150.0; Saturation gap: 33.5842</small>
- **reach**: The nearest lounge, al-jada, is 2.48 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 2.4824; Nearest lounge: al-jada</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

### al-dhaid-sharjah — **GROW** (template)

**GROW: big enough, and unsaturated.**

- **size**: About 73,067 addressable women 15+ live here, 73,067 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 73067.0302; Women 15+: 73067.0302; Size line: 20000; Big enough: True; Affluence coverage: 0.0</small>
- **saturation**: 5.43 premium reviews per 1k women, under the 150 line.  
  <small>Premium saturation: 5.4338; Premium salons: 8; Unsaturated: True; Saturation line: 150.0; Saturation gap: 144.5662</small>
- **reach**: The nearest lounge, al-jada, is 43.78 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 43.7824; Nearest lounge: al-jada</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

### kalba-sharjah — **GROW** (template)

**GROW: big enough, and unsaturated.**

- **size**: About 51,157 addressable women 15+ live here, 51,157 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 51157.3954; Women 15+: 51157.3954; Size line: 20000; Big enough: True; Affluence coverage: 0.0</small>
- **saturation**: 25.19 premium reviews per 1k women, under the 150 line.  
  <small>Premium saturation: 25.1858; Premium salons: 11; Unsaturated: True; Saturation line: 150.0; Saturation gap: 124.8142</small>
- **reach**: The nearest lounge, al-taif-mall, is 9 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 9.0015; Nearest lounge: al-taif-mall</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

### al-madam-sharjah — **GROW** (template)

**GROW: big enough, and unsaturated.**

- **size**: About 38,980 addressable women 15+ live here, 38,980 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 38980.1753; Women 15+: 38980.1753; Size line: 20000; Big enough: True; Affluence coverage: 0.0</small>
- **saturation**: 1.05 premium reviews per 1k women, under the 150 line.  
  <small>Premium saturation: 1.0507; Premium salons: 1; Unsaturated: True; Saturation line: 150.0; Saturation gap: 148.9493</small>
- **reach**: The nearest lounge, mirdif-35, is 46.72 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 46.7161; Nearest lounge: mirdif-35</small>
- **data_gap**: Our salon search covered 37% of the women here, so competition is partly unknown.  
  <small>Competitor data coverage: 0.3662; Coverage needed: 0.5; Unsaturated: True; Coverage gap: -0.1338</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

### khor-fakkan-sharjah — **GROW** (template)

**GROW: big enough, and unsaturated.**

- **size**: About 31,163 addressable women 15+ live here, 31,163 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 31162.5661; Women 15+: 31162.5661; Size line: 20000; Big enough: True; Affluence coverage: 0.0</small>
- **saturation**: 28.48 premium reviews per 1k women, under the 150 line.  
  <small>Premium saturation: 28.4775; Premium salons: 11; Unsaturated: True; Saturation line: 150.0; Saturation gap: 121.5225</small>
- **reach**: The nearest lounge, al-taif-mall, is 26.5 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 26.5027; Nearest lounge: al-taif-mall</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

...

### al-jaddaf-dubai — **WATCH** (template)

**WATCH: not big enough, and saturated.**

- **size**: About 5,647 addressable women 15+ live here, 5,647 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 5647.3902; Women 15+: 5647.3902; Size line: 20000; Big enough: False; Affluence coverage: 1.0</small>
- **saturation**: 457 premium reviews per 1k women, over the 150 line.  
  <small>Premium saturation: 456.6336; Premium salons: 6; Unsaturated: False; Saturation line: 150.0; Saturation gap: -306.6336</small>
- **reach**: The nearest lounge, city-walk, is 6.74 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 6.7442; Nearest lounge: city-walk</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

### madinat-al-riyad-abu-dhabi — **WATCH** (template)

**WATCH: not big enough, and unsaturated.**

- **size**: About 5,564 addressable women 15+ live here, 5,564 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 5564.3445; Women 15+: 5564.3445; Size line: 20000; Big enough: False; Affluence coverage: 0.0</small>
- **saturation**: 38.37 premium reviews per 1k women, under the 150 line.  
  <small>Premium saturation: 38.365; Premium salons: 2; Unsaturated: True; Saturation line: 150.0; Saturation gap: 111.635</small>
- **reach**: The nearest lounge, al-falah, is 10.25 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 10.2547; Nearest lounge: al-falah</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

### al-mizhar-2-dubai — **WATCH** (template)

**WATCH: not big enough, and saturated.**

- **size**: About 5,533 addressable women 15+ live here, 5,533 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 5533.4427; Women 15+: 5533.4427; Size line: 20000; Big enough: False; Affluence coverage: 1.0</small>
- **saturation**: 323 premium reviews per 1k women, over the 150 line.  
  <small>Premium saturation: 322.5566; Premium salons: 2; Unsaturated: False; Saturation line: 150.0; Saturation gap: -172.5566</small>
- **reach**: The nearest lounge, mirdif-35, is 3.64 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 3.6387; Nearest lounge: mirdif-35</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

### al-kheeran-dubai — **WATCH** (template)

**WATCH: not big enough, and unsaturated.**

- **size**: About 5,428 addressable women 15+ live here, 5,428 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 5427.6768; Women 15+: 5427.6768; Size line: 20000; Big enough: False; Affluence coverage: 1.0</small>
- **saturation**: 142 premium reviews per 1k women, under the 150 line.  
  <small>Premium saturation: 141.5789; Premium salons: 3; Unsaturated: True; Saturation line: 150.0; Saturation gap: 8.4211</small>
- **reach**: The nearest lounge, nad-al-sheba, is 6.63 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 6.6295; Nearest lounge: nad-al-sheba</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

### al-marmoom-dubai — **WATCH** (template)

**WATCH: not big enough, and unsaturated.**

- **size**: About 5,187 addressable women 15+ live here, 5,187 before affluence weighting (GROW needs 20,000).  
  <small>Addressable women 15+: 5187.2713; Women 15+: 5187.2713; Size line: 20000; Big enough: False; Affluence coverage: 0.1152</small>
- **saturation**: 0 premium reviews per 1k women, under the 150 line.  
  <small>Premium saturation: 0.0; Premium salons: 0; Unsaturated: True; Saturation line: 150.0; Saturation gap: 150.0</small>
- **reach**: The nearest lounge, nad-al-sheba, is 22.72 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 22.7169; Nearest lounge: nad-al-sheba</small>

_An area is big enough when enough women live there, and unsaturated when few premium-salon reviews exist per woman. GROW needs both, WATCH has one, SKIP has neither._

In [7]:
table("area", areas[0][1])

,Value,Unit,Threshold,What it means
Factor,,,,
Women 15+,"281,946",women,"SKIP under 5,000",Women aged 15+ living in the area's cells (raw).
Addressable women 15+,"281,946",women,"GROW needs 20,000","Catchment women weighted by how affluent their cell is (observed median rent vs the women-weighted median, to the af..."
Affluence rent,n/a (missing),AED/yr,—,"Median annual household rent (DLD Ejari contracts, Jul-Oct 2026), weighted by women, over the cells with an observed..."
Affluence coverage,0%,% of women,"Under 50%: affluence mostly unknown, weighted neutral",Share of the women here living in cells with an observed rent. The rest are weighted neutral: affluence unknown. Dub...
Cells,89,count,—,Populated ~2 km cells in the area (every unreached cell with this OSM place name).
Worker housing,12%,% of adults,GROW needs under 50%,Share of the area's adults living in worker housing (OSM industrial land use).
Premium salons,107,count,—,Premium salons found in the area's searched cells.
Premium saturation,116,reviews per 1k women,Unsaturated under 150,"Premium salons' Google reviews (scaled up for missed salons) per 1,000 women in the searched cells: the same measure..."
Shop rent in the area,n/a (missing),AED/m²/yr,—,Women-weighted median DLD shop rent over the area's cells (Dubai only; shown when cells with a rent hold half its wo...


## Play with it

Change a number and see the fallback: the cache key is a hash of the fact sheet, so any change
means the template is served instead of the cached AI text (and `verify` would reject the old
text anyway).

In [8]:
k, sid, a, f = next(s for s in subjects if s[1] == "al-barsha")
changed = {**f, "catchment_women": f["catchment_women"] + 1}
print(explain(k, sid, a, changed, cache=cache).source, "<- template, because one number changed")

template <- template, because one number changed
